In [299]:
from contextlib import nullcontext
from statistics import linear_regression

import numpy as np
import pandas as pd
from numba.cpython.builtins import max_iterable
from numpy.ma.core import append

pd.set_option('display.max_columns', None)

from sklearn.pipeline import  make_pipeline
from sklearn.compose import ColumnTransformer

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV

from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression

from feature_engine.imputation import  (MeanMedianImputer, ArbitraryNumberImputer, EndTailImputer, RandomSampleImputer)

In [300]:
try:

    df = pd.read_csv(r'/home/mukul/Main/Basic_DataScience-4ML/assets/data/titanic.csv',
                     usecols=['Survived','Pclass','Sex','Age','Fare','Embarked'],
                     header=0,
                     # index_col=0,
                     engine='python',
                     on_bad_lines='skip'

                     )

except Exception as e:
    print(e)

In [301]:
df.sample(5)

,Survived,Pclass,Sex,Age,Fare,Embarked
242,0,2,male,29.0,10.5000,S
371,0,3,male,18.0,6.4958,S
357,0,2,female,38.0,13.0000,S
297,0,1,female,2.0,151.5500,S
280,0,3,male,65.0,7.7500,Q


In [302]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  891 non-null    int64  
 1   Pclass    891 non-null    int64  
 2   Sex       891 non-null    object 
 3   Age       714 non-null    float64
 4   Fare      891 non-null    float64
 5   Embarked  889 non-null    object 
dtypes: float64(2), int64(2), object(2)
memory usage: 41.9+ KB


So find for Age and embarket best  imputer using gsc

In [303]:
x = df.drop('Survived', axis=1)
y = df['Survived']
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=7)

In [304]:
x_train.sample(10)

,Pclass,Sex,Age,Fare,Embarked
85,3,female,33.0,15.8500,S
176,3,male,NaN,25.4667,S
881,3,male,33.0,7.8958,S
618,2,female,4.0,39.0000,S
503,3,female,37.0,9.5875,S
23,1,male,28.0,35.5000,S
116,3,male,70.5,7.7500,Q
318,1,female,31.0,164.8667,S
659,1,male,58.0,113.2750,C
152,3,male,55.5,8.0500,S


In [305]:
class auto_imputer:
    def __init__(self,model):
        self.model = model


    def fit(self,df):

        nums_col = []
        cats_col = []
        for i in df.columns:
            if df[i].dtype == 'object':
                cats_col.append(i)
            else:
                nums_col.append(i)


        nums_pipeline = make_pipeline(MeanMedianImputer(imputation_method="mean"))
        cats_pipeline = make_pipeline(OneHotEncoder(handle_unknown='ignore'))




        cr = ColumnTransformer(transformers=[
            ('col',cats_pipeline,cats_col),
            ('num',nums_pipeline,nums_col)
         ])
        model = make_pipeline(cr, self.model)
        print(nums_pipeline.get_params())
        parameter_grid = {
            'columntransformer__num__meanmedianimputer': [
                MeanMedianImputer(imputation_method='median'),
                MeanMedianImputer(imputation_method='mean'),
                ArbitraryNumberImputer(arbitrary_number=-999),
                EndTailImputer(imputation_method='gaussian',tail='right'),
                RandomSampleImputer(random_state=42),
            ],
             'logisticregression__C': [0.1, 1, 10],
        }
        gscv = GridSearchCV(estimator=model,param_grid=parameter_grid,scoring='accuracy',cv=5,n_jobs=-1)
        gscv.fit(x_train,y_train)
        self.best_model = gscv.best_estimator_
        return print(gscv.best_params_),  print(gscv.best_score_)



In [306]:
linear = auto_imputer(LogisticRegression(max_iter=1000))
linear.fit(x_train)

{'memory': None, 'steps': [('meanmedianimputer', MeanMedianImputer(imputation_method='mean'))], 'transform_input': None, 'verbose': False, 'meanmedianimputer': MeanMedianImputer(imputation_method='mean'), 'meanmedianimputer__imputation_method': 'mean', 'meanmedianimputer__variables': None}
{'columntransformer__num__meanmedianimputer': EndTailImputer(), 'logisticregression__C': 0.1}
0.8090121146459174


(None, None)

But the above code is not dynamic and it is collective what if we name each pipeline for each col and name that pipeline on that colum and then fead that named pipeline for a deep grid search cv

In [307]:

class auto_imputer:
    def __init__(self, model):
        self.model = model

    def fit(self, x_train, y_train):

        nums_col = []
        cats_col = []

        for i in x_train.columns:
            if x_train[i].dtype == 'object':
                cats_col.append(i)
            else:
                nums_col.append(i)

        param_grid = {}
        transformers = []

        param = [
            MeanMedianImputer(imputation_method='median'),
            MeanMedianImputer(imputation_method='mean'),
            ArbitraryNumberImputer(arbitrary_number=-999),
            EndTailImputer(imputation_method='gaussian', tail='right'),
            RandomSampleImputer(random_state=42),
        ]

        for i in nums_col:
            name = i.lower()

            nums_pipeline = make_pipeline(
                MeanMedianImputer(imputation_method='mean')
            )

            transformers.append((name, nums_pipeline, [i]))

            param_grid[
                f'columntransformer__{name}__meanmedianimputer'
            ] = param

        for i in cats_col:
            name = i.lower()

            cats_pipeline = make_pipeline(
                SimpleImputer(strategy='most_frequent'),
                OneHotEncoder(handle_unknown='ignore')
            )

            transformers.append((name, cats_pipeline, [i]))

            param_grid[
                f'columntransformer__{name}__simpleimputer__strategy'
            ] = ['most_frequent', 'constant']

        cr = ColumnTransformer(transformers=transformers)

        model = make_pipeline(cr, self.model)

        gscv = GridSearchCV(
            param_grid=param_grid,
            estimator=model,
            scoring='accuracy',
            cv=5,
            n_jobs=-1
        )

        gscv.fit(x_train, y_train)

        self.best_model = gscv.best_estimator_

        for col in nums_col:
            imputer = (
                self.best_model
                .named_steps['columntransformer']
                .named_transformers_[col.lower()]
                .named_steps['meanmedianimputer']
            )

            print(f'{col}: {imputer.imputation_method}')



        return self

In [308]:
linear = auto_imputer(LogisticRegression(max_iter=1000))
linear.fit(x_train, y_train)

Pclass: median
Age: median
Fare: median


```
x_train, y_train
      |
      v
auto_imputer.fit(x_train, y_train)
      |
      v
Detect column types
      |
      +------> nums_col = [Age, Fare, Pclass]
      |
      +------> cats_col = [Sex, Embarked]
      |
      v
Build transformers list
      |
      +------> ('age', age_pipeline, ['Age'])
      +------> ('fare', fare_pipeline, ['Fare'])
      +------> ('sex', cats_pipeline, ['Sex'])
      +------> ...
      |
      v
ColumnTransformer(transformers)
      |
      v
make_pipeline(ColumnTransformer, LogisticRegression)
      |
      v
Build param_grid dictionary
      |
      +------> Age imputer candidates
      +------> Fare imputer candidates
      +------> LogisticRegression C candidates
      |
      v
GridSearchCV(model, param_grid, cv=5)
      |
      v
Try every parameter combination
      |
      v
Evaluate each combination using 5-fold CV
      |
      v
Select best combination
      |
      +------> best_params_
      +------> best_score_
      +------> best_estimator_
      |
      v
Predict on x_test

- Multivarient Imputer
    - KNN Imputer
    - Iterative Imputer

# KNN Imputer: Quick Notes

## Idea
Fill a missing value using the values of the **k most similar rows** (nearest neighbors). Similarity is measured by distance computed on the features that are not missing.

## Steps
1. Pick a row with a missing value in feature *j*.
2. Compute its distance to every other row that has feature *j* observed.
3. Select the **k nearest** rows.
4. Impute feature *j* from those neighbors' values (mean or weighted mean).
5. Repeat for every missing cell.

## Imputation Formulas

**Uniform (simple average):**

x̂ⱼ = (1/k) · Σᵢ₌₁ᵏ vᵢ

**Distance-weighted:**

x̂ⱼ = ( Σᵢ wᵢ · vᵢ ) / ( Σᵢ wᵢ ), where wᵢ = 1 / dᵢ

- vᵢ = value of feature *j* in the i-th neighbor
- dᵢ = distance to the i-th neighbor (closer neighbors get more weight)
- For categorical features, use the **mode** of the neighbors instead of the mean.

## Standard Euclidean Distance (no missing values)

d(x, y) = √( Σᵢ₌₁ⁿ (xᵢ − yᵢ)² )

## NaN-Euclidean Distance (when data is missing)

Only use coordinates that are **present in both** rows, then scale up to compensate for the skipped ones:

d(x, y) = √( (n / m) · Σ over present coords (xᵢ − yᵢ)² )

- n = total number of features
- m = number of features present in **both** x and y
- (n / m) = weight factor that compensates for the missing coordinates
- If no coordinates overlap (m = 0), the distance is NaN.

**Example:** n = 4, x = [1, NaN, 3, 5], y = [2, 4, NaN, 9]
Common coords: 1st and 4th, so m = 2.

d = √( (4/2) · [(1−2)² + (5−9)²] ) = √(2 · 17) = √34 ≈ 5.83

## Key Points
- **Scale the features first** (standardize or min-max), since distance is scale-sensitive.
- **k**: small k is noisy (high variance); large k over-smooths (high bias). Tune it with cross-validation.
- **Weights**: uniform treats all neighbors equally; distance weighting favors closer ones.
- Works best when the data has real local structure (similar rows have similar values).

## Pros
- Uses relationships between features, so it is better than mean/median imputation
- Non-parametric, with no distribution assumption
- Handles multiple missing columns

## Cons
- Slow on large datasets (computes distances to all rows), roughly O(n²·d)
- Sensitive to outliers and irrelevant features
- Needs scaling, and the choice of k matters
- Struggles in high dimensions (curse of dimensionality)

We can aply KNN, mice as alternative

multivariateimputation by chain rule

but we need old dataset
use when data is missing at random